<a href="https://colab.research.google.com/github/hazardscarn/google-tunix-kaggle/blob/dev-v1/tunix_model_pinocchio_sft_simpo_v1_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q kagglehub tensorflow datasets
!pip install "google-tunix[prod]"
!pip uninstall -q -y flax
!pip install -U flax
# !pip install -q google-generativeai

import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 830.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.0/201.0 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 140.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.9/193.9 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 159.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Using cached flax-0.12.2-py3-none-any.whl.metadata (11 kB)
Using cached flax-0.12.2-py3-none-any.whl (488 kB)


In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import numpy as np
import pandas as pd
import json
import time
from tqdm import tqdm
import orbax.checkpoint as ocp
from google.colab import drive, userdata
#import google.generativeai as genai
from datasets import load_dataset
from tunix.models.gemma3 import params, model as gemma_model
from tunix.generate import sampler as sampler_lib
import os
import jax
from flax import nnx
import orbax.checkpoint as ocp
import kagglehub

print(f"JAX devices: {jax.devices()}")
print(f"Device count: {len(jax.devices())}")
print(f"Device type: {jax.devices()[0].platform}")

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


JAX devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
Device count: 1
Device type: tpu


Login to Kaggle

In [ ]:

kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


## Parameters and Config

In [ ]:
# SHARDIN#
# MESH = [(1, 1), ("fsdp", "tp")]
num_devices = len(jax.devices())
MESH = [(1, num_devices), ("fsdp", "tp")]

model_iteration_name="pinocchio_sft_v3"

# Generation settings
#Keeping diverse values so we can mimic a PPO/GRPO here
INF_TEMPERATURE=0.9
INF_TOP_K=200
INF_TOP_P=1
NUM_GENERATIONS=4
SEED=42


## Helper Functions

In [ ]:
def load_data_from_kaggle(file_path,file_name):

  path = kagglehub.dataset_download(file_path)
  df = pd.read_parquet(f"{path}/{file_name}")
  return df

def get_gemma_ref_model(ckpt_path,model_config):
  mesh = jax.make_mesh(*MESH)
  abs_gemma: nnx.Module = nnx.eval_shape(
      lambda: params.create_model_from_checkpoint(params.GEMMA3_1B_IT, model_config)
  )

  abs_state = nnx.state(abs_gemma)
  abs_state = jax.tree.map(
      lambda a, s: jax.ShapeDtypeStruct(a.shape, jnp.bfloat16, sharding=s),
      abs_state,
      nnx.get_named_sharding(abs_state, mesh),
  )
  checkpointer = ocp.StandardCheckpointer()
  restored_params = checkpointer.restore(ckpt_path, target=abs_state)

  graph_def, _ = nnx.split(abs_gemma)
  gemma = nnx.merge(graph_def, restored_params)
  return gemma, mesh


def generate_response_batched(sampler, mesh, prompts_list, max_tokens=1024, temperature=0.7, top_k=50, top_p=0.95,seed=42):
    """
    Generate responses for MULTIPLE prompts in one call (TRUE PARALLELISM)
    """
    # 1. Format all prompts
    formatted_prompts = []
    for prompt in prompts_list:
        formatted = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
        formatted_prompts.append(formatted)

    # 2. Generate for ALL prompts at once
    # The 'mesh' context ensures the computation is distributed across TPU cores
    with mesh:
        output = sampler(
            input_strings=formatted_prompts,
            max_generation_steps=max_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            eos_tokens=[1, 106], # 1=EOS, 106=<end_of_turn>
            #seed=seed,
            seed=seed if seed is not None else random.randint(0, 2**31-1),
        )

    # 3. Extract clean responses
    clean_responses = []

    # output.text is guaranteed to be a list of strings matching the input order
    for full_response in output.text:
        if '<end_of_turn>' in full_response:
            clean = full_response.split('<end_of_turn>')[0] + '<end_of_turn>'
        else:
            clean = full_response
        clean_responses.append(clean)

    return clean_responses


print("\n" + "="*60)
print("Running BATCHED Inference")
print("="*60)


def run_batched_generation(eval_df, BATCH_SIZE, temperature, top_k, top_p, seed=42, max_tokens=1600):
  all_responses = []
  all_errors = []
  prompts = eval_df['input'].tolist()

  for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc="Batched generation"):
      batch_prompts = prompts[i:i+BATCH_SIZE]

      try:
          batch_responses = generate_response_batched(
              sampler=sampler,
              mesh=mesh,
              prompts_list=batch_prompts,
              max_tokens=max_tokens,
              temperature=temperature,
              top_k=top_k,
              top_p=top_p,
          )
          all_responses.extend(batch_responses)
          all_errors.extend([None] * len(batch_responses))

      except Exception as e:
          print(f"\nError in batch {i//BATCH_SIZE}: {str(e)}")

          # === FIX START ===
          # We MUST add placeholders to keep lists the same length as the dataframe
          current_batch_len = len(batch_prompts)
          all_responses.extend(["ERROR_CACHE_LIMIT"] * current_batch_len)
          all_errors.extend([str(e)] * current_batch_len)
          # === FIX END ===

  # Now lengths are guaranteed to match
  eval_df['student_response'] = all_responses
  eval_df['error'] = all_errors

  return eval_df

  # # Statistics
  # successful = sum(1 for e in all_errors if e is None)
  # failed = sum(1 for e in all_errors if e is not None)

  # print(f"\n✓ Batched inference complete")
  # print(f"  Total prompts: {len(eval_df)}")
  # print(f"  Processed in: {len(prompts)//BATCH_SIZE + 1} batches")
  # print(f"  Successful: {successful}/{len(eval_df)}")
  # print(f"  Failed: {failed}/{len(eval_df)}")
  # print("\n" + "="*60)
  # print("Evaluation Complete!")
  # print("="*60)

  # return eval_df


def generate_multiple_responses_per_prompt(
    transformer,
    tokenizer,
    model_config,
    mesh,
    eval_df,
    num_samples=3,
    batch_size=32,
    max_tokens=1600,
    temperature=0.8,
    top_k=50,
    top_p=0.95
):
    """
    Generate N responses for each prompt using the WORKING pattern from generate_multiple_for_analysis.
    Key: Create fresh sampler for EACH BATCH, not just each sample.
    """

    all_results = []
    prompts = eval_df['input'].tolist()

    cache_config = sampler_lib.CacheConfig(
        cache_size=4096,
        num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads,
        head_dim=model_config.head_dim,
    )

    # Generate responses for each sample_id
    for sample_id in range(num_samples):
        print(f"\n{'='*60}")
        print(f"Generating Sample {sample_id + 1}/{num_samples}")
        print(f"{'='*60}")

        sample_responses = []
        sample_errors = []

        # Process in batches
        for i in tqdm(range(0, len(prompts), batch_size), desc=f"Sample {sample_id}"):
            batch_prompts = prompts[i:i+batch_size]

            # CREATE FRESH SAMPLER FOR EACH BATCH (like generate_multiple_for_analysis does)
            fresh_sampler = sampler_lib.Sampler(
                transformer=transformer,
                tokenizer=tokenizer,
                cache_config=cache_config,
            )

            try:
                batch_responses = generate_response_batched(
                    sampler=fresh_sampler,
                    mesh=mesh,
                    prompts_list=batch_prompts,
                    max_tokens=max_tokens,
                    temperature=temperature,
                    top_k=top_k,
                    top_p=top_p,
                    seed=sample_id,  # Use sample_id as seed (like resp in your working function)
                )
                sample_responses.extend(batch_responses)
                sample_errors.extend([None] * len(batch_responses))

            except Exception as e:
                print(f"\nError in batch {i//batch_size}: {str(e)}")
                current_batch_len = len(batch_prompts)
                sample_responses.extend([""] * current_batch_len)
                sample_errors.extend([str(e)] * current_batch_len)

            # Clean up sampler immediately after use
            del fresh_sampler

        # Create rows for this sample
        for idx, (uid, prompt, response, error) in enumerate(
            zip(eval_df.index, prompts, sample_responses, sample_errors)
        ):
            all_results.append({
                'uid': uid,
                'input': prompt,
                'sample_id': sample_id,
                'student_response': response,
                'error': error
            })

    results_df = pd.DataFrame(all_results)

    # Statistics
    total = len(results_df)
    successful = (results_df['student_response'] != "").sum()
    failed = (results_df['student_response'] == "").sum()

    print(f"\n{'='*60}")
    print("Multi-Response Generation Complete!")
    print(f"{'='*60}")
    print(f"Total prompts: {len(prompts)}")
    print(f"Samples per prompt: {num_samples}")
    print(f"Total responses: {total}")
    print(f"Successful: {successful}/{total} ({100*successful/total:.1f}%)")
    print(f"Failed: {failed}/{total}")

    return results_df





Running BATCHED Inference


## Load the Model to do inference on

In [ ]:
from tunix.models.gemma3 import params_safetensors as params_safetensors_lib

# 1. Download the Model
# Replace with your handle if you used a different one
HANDLE = "davidacad10/Pinocchio_SFT/flax/default"
print(f"Downloading {HANDLE}...")
model_path = kagglehub.model_download(HANDLE)
print(f"Model downloaded to: {model_path}")

# 2. Define Architecture Config
# Since SFT keeps the architecture same, we load the standard 1B config
model_config = gemma_model.ModelConfig.gemma3_1b()
# 3. Create JAX Mesh
# Standard mesh for 1B model (Data Parallelism usually sufficient for inference on 1 chip)
# If using TPU with 8 cores, we can split across them.
device_count = len(jax.devices())
mesh = jax.make_mesh(
    (1, device_count),
    ("fsdp", "tp"),
    axis_types=(jax.sharding.AxisType.Auto,) * 2
)

# 4. Load the Model
print("Loading weights from SafeTensors...")
sft_model = params_safetensors_lib.create_model_from_safe_tensors(
    file_dir=model_path,
    config=model_config,
    mesh=mesh,
    dtype=jnp.bfloat16 # Ensure bfloat16 for TPU/Ampere GPUs
)


100%|██████████| 1.20k/1.20k [00:00<00:00, 2.05MB/s]



100%|██████████| 465/465 [00:00<00:00, 797kB/s]



  0%|          | 0.00/1.86G [00:00<?, ?B/s]
  0%|          | 1.00M/1.86G [00:00<12:48, 2.60MB/s]
  0%|          | 3.00M/1.86G [00:00<04:38, 7.16MB/s]
  0%|          | 8.00M/1.86G [00:00<01:44, 19.0MB/s]
  1%|          | 14.0M/1.86G [00:00<01:05, 30.1MB/s]
  1%|          | 18.0M/1.86G [00:01<01:28, 22.4MB/s]
  1%|          | 23.0M/1.86G [00:01<01:18, 25.3MB/s]
  1%|▏         | 27.0M/1.86G [00:01<01:17, 25.4MB/s]
  2%|▏         | 34.0M/1.86G [00:01<00:58, 33.3MB/s]
  2%|▏         | 39.0M/1.86G [00:01<00:52, 37.1MB/s]
  2%|▏         | 43.0M/1.86G [00:01<00:59, 32.8MB/s]
  3%|▎         | 48.0M/1.86G [00:01<00:54, 35.7MB/s]
  3%|▎         | 53.0M/1.86G [00:01<00:50, 38.4MB/s]
  3%|▎         | 59.0M/1.86G [00:02<00:51, 37.3MB/s]
  3%|▎         | 66.0M/1.86G [00:02<00:53, 36.0MB/s]
  4%|▍         | 73.0M/1.86G [00:02<00:51, 37.3MB/s]
  4%|▍         | 80.0M/1.86G [00:02<00:45, 42.0MB/s]
  4%|▍         | 85.0M/1.86G [00:02<00:46, 40.8MB/s]
  5%|▍         | 92.0M/1.86G [00:02<00:42, 45.0MB/s]


Model downloaded to: /root/.cache/kagglehub/models/davidacad10/Pinocchio_SFT/flax/default/1
Loading weights from SafeTensors...


In [ ]:
from random import seed
tokenizer = params.create_tokenizer()
sampler = sampler_lib.Sampler(
    transformer=sft_model,
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=4096,
        num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads,
        head_dim=model_config.head_dim,
    ),
)


### Load the Perfect Teacher Sample that's base data for DPO

In [ ]:
gen_base = pd.read_parquet('/content/dpo_base_from_teacher_with_batch.parquet').reset_index(drop=True)

# # Shuffle the data
# gen_base = gen_base.sample(frac=1).reset_index(drop=True)

# # Create 10 different batches
# num_batches = 10
# records_per_batch = len(gen_base) // num_batches

# batch_assignments = []
# for i in range(num_batches):
#     batch_assignments.extend([f'batch {i}'] * records_per_batch)

# # Handle any remaining records if the total number of records is not perfectly divisible
# remaining_records = len(gen_base) % num_batches
# if remaining_records > 0:
#     # Distribute remaining records to existing batches
#     for i in range(remaining_records):
#         batch_assignments.append(f'batch {i}')

# # Ensure the length of batch_assignments matches the DataFrame length
# # This might happen if remaining_records is large and leads to index out of bounds in extend if not careful
# # A simpler way to handle this, after extending, is to trim or pad if needed, but given the approach, it should be fine.
# # However, it's safer to generate a list of batch assignments that exactly matches the df length
# batch_column = np.repeat(np.arange(num_batches), records_per_batch).tolist()
# # Add remaining items if any
# batch_column.extend(np.arange(remaining_records).tolist())

# gen_base['batch'] = [f'batch_{b}' for b in batch_column]

# # Display the count of records per batch to verify distribution
# display(gen_base['batch'].value_counts())
# gen_base.to_parquet('/content/dpo_base_from_teacher_with_batch.parquet')

In [ ]:
display(gen_base['batch'].value_counts())

,count
batch,
batch_0,3132
batch_1,3131
batch_2,3131
batch_3,3131
batch_4,3131
batch_5,3131
batch_6,3131
batch_7,3131
batch_8,3131


##Create Sampler For the Model

In [ ]:
from random import seed
sampler = sampler_lib.Sampler(
    transformer=sft_model,
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=4096,
        num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads,
        head_dim=model_config.head_dim,
    ),
)


In [ ]:
print("DPO Base Sample Distibution")
print(gen_base.groupby('domain').size())

DPO Base Sample Distibution
domain
MultiTask Knowledge               2168
code                              1574
commonsense_reasoning             1499
conversational                    1278
creative_ideation                  883
creative_writing                    62
financial_reasoning               1185
math                              8100
numerical_reasoning               1555
reading_comprehension             1520
roleplay                          3509
safety_and_ethics                 2292
science                           1228
scientific_understanding          1354
summarization                      283
table_extraction                  1490
trick_questions_misconceptions    1331
dtype: int64


### Let's generate the SFT trained student models reponse for each batch of samples separately

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
batch1_df= gen_base[gen_base['batch']=='batch_0'].copy()
batch1_response = generate_multiple_responses_per_prompt(
    transformer=sft_model,
    tokenizer=tokenizer,
    model_config=model_config,
    mesh=mesh,
    eval_df=batch1_df,
    num_samples=3,  # Generate A, B, C for each prompt
    batch_size=64,
    temperature=1.2,
    top_k=200,
    top_p=1,
    # Higher temp for diversity
    max_tokens=1792,
)

batch1_response.to_csv('batch1_multi_response.csv',index=False)
output_path = '/content/gdrive/MyDrive/batch1_multi_response.csv'
batch1_response.to_csv(output_path, index=False)
print(f'batch1_multi_response.csv saved to {output_path}')


Generating Sample 1/3


Sample 0: 100%|██████████| 49/49 [1:46:10<00:00, 130.00s/it]



Generating Sample 2/3


Sample 1: 100%|██████████| 49/49 [1:45:40<00:00, 129.39s/it]



Generating Sample 3/3


Sample 2: 100%|██████████| 49/49 [1:45:24<00:00, 129.08s/it]



Multi-Response Generation Complete!
Total prompts: 3132
Samples per prompt: 3
Total responses: 9396
Successful: 9396/9396 (100.0%)
Failed: 0/9396


In [ ]:
# rest_batch= gen_base[gen_base['batch']!='batch_0'].copy()
# rest_batch = rest_batch[rest_batch['domain'].isin(['numerical_reasoning',
#                                                    'financial_reasoning',
#                                                    'reading_comprehension',
#                                                    'science',
#                                                    'scientific_understanding',
#                                                    'table_extraction',
#                                                    'trick_questions_misconceptions',
#                                                    'safety_and_ethics',
#                                                    'creative_ideation',
#                                                    'creative_writing',
#                                                    'summarization'])]
# rest_batch_response = generate_multiple_responses_per_prompt(
#     transformer=sft_model,
#     tokenizer=tokenizer,
#     model_config=model_config,
#     mesh=mesh,
#     eval_df=rest_batch,
#     num_samples=3,  # Generate A, B, C for each prompt
#     batch_size=64,
#     temperature=1.2,
#     top_k=200,
#     top_p=1,
#     # Higher temp for diversity
#     max_tokens=1792,
# )

# rest_batch_response.to_csv('restbatch_multi_response.csv',index=False)

In [ ]:
batch2_df= gen_base[gen_base['batch']=='batch_1'].copy()
batch2_df = batch2_df[~batch2_df['domain'].isin(['math','code','conversational','roleplay'])]

batch2_response = generate_multiple_responses_per_prompt(
    transformer=sft_model,
    tokenizer=tokenizer,
    model_config=model_config,
    mesh=mesh,
    eval_df=batch2_df,
    num_samples=3,  # Generate A, B, C for each prompt
    batch_size=64,
    temperature=1.2,
    top_k=200,
    top_p=1,
    # Higher temp for diversity
    max_tokens=1792,
)

output_path = '/content/gdrive/MyDrive/batch2_multi_response.csv'
batch2_response.to_csv(output_path, index=False)
print(f'batch2_multi_response.csv saved to {output_path}')


Generating Sample 1/3


Sample 0: 100%|██████████| 27/27 [54:32<00:00, 121.21s/it]



Generating Sample 2/3


Sample 1: 100%|██████████| 27/27 [53:57<00:00, 119.91s/it]



Generating Sample 3/3


Sample 2: 100%|██████████| 27/27 [55:49<00:00, 124.06s/it]



Multi-Response Generation Complete!
Total prompts: 1724
Samples per prompt: 3
Total responses: 5172
Successful: 5172/5172 (100.0%)
Failed: 0/5172
batch2_multi_response.csv saved to /content/gdrive/MyDrive/batch2_multi_response.csv


In [ ]:
batch3_df= gen_base[gen_base['batch']=='batch_2'].copy()
batch3_df = batch3_df[~batch3_df['domain'].isin(['math','code','conversational','roleplay'])]

batch3_response = generate_multiple_responses_per_prompt(
    transformer=sft_model,
    tokenizer=tokenizer,
    model_config=model_config,
    mesh=mesh,
    eval_df=batch3_df,
    num_samples=3,  # Generate A, B, C for each prompt
    batch_size=64,
    temperature=1.2,
    top_k=200,
    top_p=1,
    # Higher temp for diversity
    max_tokens=1792,
)

output_path = '/content/gdrive/MyDrive/batch3_multi_response.csv'
batch3_response.to_csv(output_path, index=False)
print(f'batch3_multi_response.csv saved to {output_path}')


Generating Sample 1/3


Sample 0: 100%|██████████| 27/27 [55:03<00:00, 122.35s/it]



Generating Sample 2/3


Sample 1: 100%|██████████| 27/27 [54:35<00:00, 121.33s/it]



Generating Sample 3/3


Sample 2: 100%|██████████| 27/27 [54:40<00:00, 121.51s/it]



Multi-Response Generation Complete!
Total prompts: 1692
Samples per prompt: 3
Total responses: 5076
Successful: 5076/5076 (100.0%)
Failed: 0/5076
batch3_multi_response.csv saved to /content/gdrive/MyDrive/batch3_multi_response.csv


In [ ]:
rest_batch= gen_base[~gen_base['batch'].isin(['batch_0','batch_1','batch_2'])].copy()
rest_batch = rest_batch[rest_batch['domain'].isin(['numerical_reasoning',
                                                   'financial_reasoning',
                                                   'reading_comprehension',
                                                   'science',
                                                   'scientific_understanding',
                                                   'table_extraction',
                                                   'trick_questions_misconceptions',
                                                   'safety_and_ethics',
                                                   'creative_ideation',
                                                   'creative_writing',
                                                   'summarization'])]
##Down sample to 5K records
rest_batch = rest_batch.sample(n=5000, random_state=42)
rest_batch_response = generate_multiple_responses_per_prompt(
    transformer=sft_model,
    tokenizer=tokenizer,
    model_config=model_config,
    mesh=mesh,
    eval_df=rest_batch,
    num_samples=3,  # Generate A, B, C for each prompt
    batch_size=64,
    temperature=1.2,
    top_k=200,
    top_p=1,
    # Higher temp for diversity
    max_tokens=1792,
)

output_path = '/content/gdrive/MyDrive/rest_multi_response.csv'
rest_batch_response.to_csv(output_path, index=False)


Generating Sample 1/3


Sample 0: 100%|██████████| 79/79 [2:39:12<00:00, 120.92s/it]



Generating Sample 2/3


Sample 1: 100%|██████████| 79/79 [2:39:16<00:00, 120.97s/it]



Generating Sample 3/3


Sample 2: 100%|██████████| 79/79 [2:37:33<00:00, 119.66s/it]



Multi-Response Generation Complete!
Total prompts: 5000
Samples per prompt: 3
Total responses: 15000
Successful: 15000/15000 (100.0%)
Failed: 0/15000


In [ ]:
rest_batch_others= gen_base[~gen_base['batch'].isin(['batch_0','batch_1','batch_2'])].copy()
rest_batch_others = rest_batch_others[rest_batch_others['domain'].isin(['math','code','MultiTask Knowledge'])]
##Down sample to 3K records
rest_batch_others = rest_batch_others.sample(n=3000, random_state=42)

rest_batch_others_response = generate_multiple_responses_per_prompt(
    transformer=sft_model,
    tokenizer=tokenizer,
    model_config=model_config,
    mesh=mesh,
    eval_df=rest_batch_others,
    num_samples=3,  # Generate A, B, C for each prompt
    batch_size=64,
    temperature=1.2,
    top_k=200,
    top_p=1,
    # Higher temp for diversity
    max_tokens=1792,
)

output_path = '/content/gdrive/MyDrive/rest_batch_others_response.csv'
rest_batch_others_response.to_csv(output_path, index=False)


Generating Sample 1/3


Sample 0: 100%|██████████| 47/47 [1:42:09<00:00, 130.41s/it]



Generating Sample 2/3


Sample 1: 100%|██████████| 47/47 [1:41:54<00:00, 130.09s/it]



Generating Sample 3/3


Sample 2: 100%|██████████| 47/47 [1:42:21<00:00, 130.66s/it]



Multi-Response Generation Complete!
Total prompts: 3000
Samples per prompt: 3
Total responses: 9000
Successful: 9000/9000 (100.0%)
Failed: 0/9000
